# FIG05: The human/bovine homology challenge (and trout), from the site-42 DIA-NN data.

```
FIG05: The human/bovine homology challenge (and trout), from the site-42 DIA-NN data.

KDE density of observed log2(A/C) coloured by peptide HOMOLOGY CLASS, with dashed
design-expected reference lines. Because human (45->3 %) and cow (5->47 %) sum to a constant
50 % in every sample, peptides SHARED between human and bovine carry the combined signal and
collapse toward ratio 1 (log2 0) -- so they cannot resolve the two species and bias protein
roll-up. Peptides UNIQUE to one species sit at their expected ratios (human +3.9, cow -3.2,
trout 0). This is the density view from bin/06_homology/peptide_ratios.ipynb.

HOMOLOGY CLASS is defined at the SEQUENCE level: an in-silico tryptic digest of the combined
FASTA records, for each observed peptide, which organisms' proteomes contain it. (We do NOT
use DIA-NN's Protein.Names for this -- DIA-NN collapses shared peptides onto a single protein
group, which mislabels ~2/3 of human/bovine-shared peptides as unique and destroys the
effect.) The peptide->class map is cached to data/fig5_peptide_category.csv.

(Panel B in the paper -- Skyline chromatograms of two single-amino-acid-variant (SAAV)
 peptides differing by one residue -- is an instrument screenshot added manually.)

Input: DIA-NN report.pr_matrix.tsv (same site-42 run as Fig 2).
Colours: Human orange, Cow green, Trout blue; human+bovine-shared purple.
```

In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

DATA42 = r"D:/2022 Multi-Species Standard Study/42"
INPUT = os.path.join(DATA42, "DIANN_out", "report.pr_matrix.tsv")
FASTA = r"D:/2022 Multi-Species Standard Study/fasta/LakeTrout-Human-Cow-Contaminants-sPRG2022.fasta"
PEPTIDE_COL = "Stripped.Sequence"
COLS = {"A": r"D:\2022 Multi-Species Standard Study\42\EncyclopeDIA\42_exploris480_DIA_A.mzML",
        "C": r"D:\2022 Multi-Species Standard Study\42\EncyclopeDIA\42_exploris480_DIA_C.mzML"}

OUTPUT = "output"
DATA = "data"
os.makedirs(OUTPUT, exist_ok=True)
os.makedirs(DATA, exist_ok=True)

COLORS = {"Human": "#E69F00", "Bovine": "#009E73", "Trout": "#56B4E9"}
ORGBIT = {"Human": 1, "Bovine": 2, "Trout": 4}
CAT = {1: "Human only", 2: "Bovine only", 4: "Trout only",
       3: "Human+Bovine", 5: "Human+Trout", 6: "Bovine+Trout", 7: "All three"}
EXPECTED_AC = {"Human": np.log2(45 / 3), "Bovine": np.log2(5 / 47), "Trout": 0.0}

CLASSES = ["Human only", "Bovine only", "Trout only", "Human+Bovine"]
CCOLOR = {"Human only": COLORS["Human"], "Bovine only": COLORS["Bovine"],
          "Trout only": COLORS["Trout"], "Human+Bovine": "#CC79A7"}

In [2]:
def clean_pep(p):
    return re.sub(r"[^A-Z]", "", re.sub(r"\[.*?\]", "", str(p).upper()))

In [3]:
def digest(seq, missed=2, min_len=6, max_len=50):
    seq = re.sub(r"[^A-Z]", "", seq.upper())
    cuts = [0] + [i + 1 for i in range(len(seq) - 1) if seq[i] in "KR" and seq[i + 1] != "P"] + [len(seq)]
    cuts = sorted(set(cuts))
    peps = set()
    for i in range(len(cuts) - 1):
        for m in range(missed + 1):
            j = i + 1 + m
            if j >= len(cuts):
                break
            p = seq[cuts[i]:cuts[j]]
            if min_len <= len(p) <= max_len:
                peps.add(p)
    return peps

In [4]:
def fasta_org(header):
    tok = header.split()[0]
    if "Cont_" in tok:
        return None          # contaminant -- not part of the 3-proteome design
    if "_HUMAN" in tok:
        return "Human"
    if "_BOVIN" in tok:
        return "Bovine"
    return "Trout"           # trout is UniProt (_SALNM) in the study FASTA

In [5]:
def classify(targets):
    cache = os.path.join(DATA, "fig5_peptide_category.csv")
    if os.path.exists(cache):
        c = pd.read_csv(cache)
        if set(targets).issubset(set(c["peptide"].astype(str))):
            return c
        print("  cache does not cover current peptides -> rebuilding")
    member = {p: 0 for p in targets}

    def flush(header, parts):
        if not header:
            return
        org = fasta_org(header)
        if org is None:                     # skip contaminants
            return
        bit = ORGBIT[org]
        for p in digest("".join(parts)):
            if p in member:
                member[p] |= bit

    header, parts = None, []
    with open(FASTA, encoding="utf-8", errors="ignore") as fh:
        for line in fh:
            if line.startswith(">"):
                flush(header, parts)
                header, parts = line[1:].strip(), []
            else:
                parts.append(line.strip())
        flush(header, parts)

    df = pd.DataFrame({"peptide": list(member), "bitmask": list(member.values())})
    df["category"] = df["bitmask"].map(lambda b: CAT.get(b, "Unassigned"))
    df.to_csv(cache, index=False)
    return df

In [6]:
def load():
    q = pd.read_csv(INPUT, sep="\t", engine="python", on_bad_lines="skip")
    q.columns = [str(c).strip() for c in q.columns]
    q["peptide"] = q[PEPTIDE_COL].map(clean_pep)
    for k, col in COLS.items():
        q[k] = pd.to_numeric(q[col], errors="coerce")
    q = q.groupby("peptide", as_index=False)[["A", "C"]].sum(min_count=1)
    cat = classify(set(q["peptide"]))
    m = q.merge(cat[["peptide", "category"]], on="peptide", how="left")
    m = m[(m["A"] > 0) & (m["C"] > 0)].copy()
    m["log2_AC"] = np.log2(m["A"] / m["C"])
    return m

In [7]:
def figure(m, out_png, min_pts=30):
    vals = m["log2_AC"].replace([np.inf, -np.inf], np.nan).dropna()
    x_lo, x_hi = np.nanpercentile(vals, [0.5, 99.5])
    grid = np.linspace(x_lo - 0.5, x_hi + 0.5, 512)

    fig, ax = plt.subplots(figsize=(7.8, 5.0))
    for cls in CLASSES:
        v = m.loc[m["category"] == cls, "log2_AC"].replace([np.inf, -np.inf], np.nan).dropna().values
        if len(v) < min_pts:
            continue
        ax.plot(grid, gaussian_kde(v)(grid), lw=2.6, alpha=0.9,
                color=CCOLOR[cls], label=f"{cls}  (n={len(v):,})")
        ax.fill_between(grid, gaussian_kde(v)(grid), color=CCOLOR[cls], alpha=0.08)
    for sp in ("Human", "Bovine", "Trout"):
        ax.axvline(EXPECTED_AC[sp], ls="--", lw=1.2, color=COLORS[sp], zorder=1)
    ax.axvline(0.0, ls=":", lw=1.4, color="#CC79A7", zorder=1)

    ax.set_xlim(grid[0], grid[-1])
    ax.set_xlabel("observed log2(A/C)")
    ax.set_ylabel("density")
    ax.set_title("FIG05  Human/bovine homology challenge (site 42, DIA-NN)", fontsize=12)
    ax.grid(True, lw=0.3)
    ax.legend(frameon=False, fontsize=9, loc="upper center", title="peptide homology class")
    fig.tight_layout()
    fig.savefig(out_png, dpi=200)
    fig.savefig(out_png.replace(".png", ".pdf"))
    plt.close(fig)
    print(f"  wrote {out_png}")

In [8]:
# ---- generate figure ----
m = load()
m.to_csv(os.path.join(DATA, "fig5_homology_long.csv"), index=False)
print("peptides by homology class:")
print(m["category"].value_counts().to_string())
print("\nmedian log2(A/C) by class:")
for cls in CLASSES:
    v = m.loc[m["category"] == cls, "log2_AC"]
    if len(v):
        print(f"  {cls:14} median={np.median(v):+.2f} (2^={2**np.median(v):.2f}), n={len(v)}")
figure(m, os.path.join(OUTPUT, "FIG05_homology_challenge.png"))

peptides by homology class:
category
Human+Bovine    4242
Trout only      2767
Bovine only     2430
All three       2115
Human only      1503
Unassigned       172
Human+Trout       62
Bovine+Trout      47

median log2(A/C) by class:
  Human only     median=+3.81 (2^=14.00), n=1503
  Bovine only    median=-3.26 (2^=0.10), n=2430
  Trout only     median=+0.00 (2^=1.00), n=2767
  Human+Bovine   median=-0.12 (2^=0.92), n=4242


  wrote output\FIG05_homology_challenge.png
